# ETL da camada silver para camada gold


## Extract

In [9]:
import pandas as pd
import psycopg
from psycopg import connect, sql
import sys
import warnings

warnings.filterwarnings('ignore')

print("--- Iniciando processo de Extract do banco ---")

DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
DB_SCHEMA = "silver"
TABLE_NAME = "listings"

TABLE_FULL_NAME = sql.SQL("{}.{}").format(
    sql.Identifier(DB_SCHEMA),
    sql.Identifier(TABLE_NAME)
)

query_object = sql.SQL("SELECT * FROM {}").format(TABLE_FULL_NAME)

connection_string = f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}"

try:
    print("Estabelecendo conexão...")
    
    with connect(connection_string) as conn:
        print("Conexão estabelecida.")
        
        query_string = query_object.as_string(conn)
        print(f"Executando query: {query_string}")

        df = pd.read_sql_query(query_string, conn)

    print("\nDados carregados do banco para o DataFrame com sucesso!")
    print(f"Total de linhas carregadas: {len(df)}")
except psycopg.Error as e:
    print(f"\n--- Ocorreu um erro ao conectar ou ler o banco de dados ---")
    print(f"Erro: {e}")
    sys.exit(1)
except Exception as e:
    print(f"\n--- Ocorreu um erro inesperado ---")
    print(f"Erro: {e}")
    sys.exit(1)

--- Iniciando processo de Extract do banco ---
Estabelecendo conexão...
Conexão estabelecida.
Executando query: SELECT * FROM "silver"."listings"

Dados carregados do banco para o DataFrame com sucesso!
Total de linhas carregadas: 99449


In [10]:
df.head(3)

,id,host_id,name,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,instant_bookable,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,has_house_rules
0,1001254,80014485718,Clean & quiet apt home by the park,False,Madaline,Brooklyn,Kensington,40.64749,-73.97237,False,...,193.0,10,9,2021-10-19,0.21,4.0,6,286,Clean up and treat the home the way you'd like...,True
1,1002102,52335172823,Skylit Midtown Castle,True,Jenna,Manhattan,Midtown,40.75362,-73.98377,False,...,28.0,30,45,2022-05-21,0.38,4.0,2,228,Pet friendly but please confirm with me if the...,True
2,1002403,78829239556,THE VILLAGE OF HARLEM....NEW YORK !,True,Elise,Manhattan,Harlem,40.80902,-73.94190,True,...,124.0,3,0,None,NaN,5.0,1,352,"I encourage you to use my kitchen, cooking and...",True


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99449 entries, 0 to 99448
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              99449 non-null  int64  
 1   host_id                         99449 non-null  int64  
 2   name                            99449 non-null  object 
 3   host_identity_verified          99449 non-null  bool   
 4   host_name                       99449 non-null  object 
 5   neighbourhood_group             99449 non-null  object 
 6   neighbourhood                   99449 non-null  object 
 7   lat                             99449 non-null  float64
 8   long                            99449 non-null  float64
 9   instant_bookable                99449 non-null  bool   
 10  cancellation_policy             99396 non-null  object 
 11  room_type                       99449 non-null  object 
 12  construction_year               

## Transform

In [4]:
cols_fato = [
    'id',
    'name',
    'room_type',
    'minimum_nights',
    'cancellation_policy',
    'instant_bookable',
    'availability_365',
    'has_house_rules',
    'construction_year'
]

df_fato = df[cols_fato]
df_fato

,id,name,room_type,minimum_nights,cancellation_policy,instant_bookable,availability_365,has_house_rules,construction_year
0,1001254,Clean & quiet apt home by the park,Private room,10,strict,False,286,True,2020
1,1002102,Skylit Midtown Castle,Entire home/apt,30,moderate,False,228,True,2007
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,Private room,3,flexible,True,352,True,2005
3,1002755,Sem nome informado,Entire home/apt,30,moderate,True,322,False,2005
4,1003689,Entire Apt: Spacious Studio/Loft by central park,Entire home/apt,10,moderate,False,289,True,2009
...,...,...,...,...,...,...,...,...,...
99444,57358028,"Room in Queens, NY, near LGA.",Private room,1,strict,True,361,True,2022
99445,57358580,Cozy home away from home,Private room,1,moderate,True,324,False,2020
99446,57359133,Central Park Views - Private Room & Bathroom,Private room,1,strict,False,0,True,2012
99447,57359685,Ultimate 50th Floor Downtown Penthouse - 4000...,Entire home/apt,2,flexible,False,343,True,2020


In [5]:
cols_preco = ['price', 'service_fee', 'id']
df_preco = df[cols_preco]
df_preco

,price,service_fee,id
0,966.0,193.0,1001254
1,142.0,28.0,1002102
2,620.0,124.0,1002403
3,368.0,74.0,1002755
4,204.0,41.0,1003689
...,...,...,...
99444,982.0,196.0,57358028
99445,946.0,189.0,57358580
99446,706.0,141.0,57359133
99447,1043.0,209.0,57359685


In [6]:
cols_avaliacao = [
    'id',
    'last_review',
    'reviews_per_month',
    'number_of_reviews',
    'review_rate_number'
]
df_avaliacao = df[cols_avaliacao]

df_avaliacao

,id,last_review,reviews_per_month,number_of_reviews,review_rate_number
0,1001254,2021-10-19,0.21,9,4.0
1,1002102,2022-05-21,0.38,45,4.0
2,1002403,None,NaN,0,5.0
3,1002755,2019-07-05,4.64,270,4.0
4,1003689,2018-11-19,0.10,9,3.0
...,...,...,...,...,...
99444,57358028,2019-06-29,8.58,239,2.0
99445,57358580,2019-06-27,2.84,76,1.0
99446,57359133,2017-08-15,0.14,4,4.0
99447,57359685,2019-07-01,0.74,21,4.0


In [7]:
cols_host = [
    'id',
    'host_id',
    'host_name',
    'host_identity_verified',
    'calculated_host_listings_count'
]
df_host = df[cols_host]

df_host

,id,host_id,host_name,host_identity_verified,calculated_host_listings_count
0,1001254,80014485718,Madaline,False,6
1,1002102,52335172823,Jenna,True,2
2,1002403,78829239556,Elise,True,1
3,1002755,85098326012,Garry,False,1
4,1003689,92037596077,Lyndon,True,1
...,...,...,...,...,...
99444,57358028,56457739998,Sonia,True,2
99445,57358580,60176837202,Sem nome informado,True,1
99446,57359133,68411243647,Sem nome informado,True,1
99447,57359685,95625271612,Sem nome informado,True,2


In [8]:
cols_location = [
    'id',
    'lat',
    'long',
    'neighbourhood',
    'neighbourhood_group',
]

df_location = df[cols_location]
df_location

,id,lat,long,neighbourhood,neighbourhood_group
0,1001254,40.64749,-73.97237,Kensington,Brooklyn
1,1002102,40.75362,-73.98377,Midtown,Manhattan
2,1002403,40.80902,-73.94190,Harlem,Manhattan
3,1002755,40.68514,-73.95976,Clinton Hill,Brooklyn
4,1003689,40.79851,-73.94399,East Harlem,Manhattan
...,...,...,...,...,...
99444,57358028,40.76245,-73.87938,East Elmhurst,Queens
99445,57358580,40.59380,-73.77373,Edgemere,Queens
99446,57359133,40.79712,-73.96117,Upper West Side,Manhattan
99447,57359685,40.72318,-74.00223,SoHo,Manhattan


## Load

In [14]:
import pandas as pd
import psycopg
from psycopg import connect, sql
import sys
import warnings

warnings.filterwarnings('ignore')

# --- 1. CONFIGURAÇÕES DO BANCO ---
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
DB_SCHEMA_SILVER = "silver"
DB_SCHEMA_GOLD = "gold"
TABLE_NAME_SILVER = "listings"

connection_string = f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}"

# --- 2. EXTRACT (Carregar dados da Silver) ---

print("--- Iniciando processo de Extract (Camada Silver) ---")

TABLE_FULL_NAME_SILVER = sql.SQL("{}.{}").format(
    sql.Identifier(DB_SCHEMA_SILVER),
    sql.Identifier(TABLE_NAME_SILVER)
)
query_object = sql.SQL("SELECT * FROM {}").format(TABLE_FULL_NAME_SILVER)

try:
    with connect(connection_string) as conn:
        print("Conexão estabelecida.")
        query_string = query_object.as_string(conn)
        # Carrega o DataFrame da Silver
        df = pd.read_sql_query(query_string, conn)
    print(f"Dados carregados da Silver. Total de {len(df)} linhas.")
except Exception as e:
    print(f"--- Erro no EXTRACT --- \nErro: {e}")
    sys.exit(1)


# --- 3. TRANSFORM & LOAD (Camada Gold) ---

print("\n--- Iniciando processo de Transform & Load (Camada Gold) ---")

try:
    # Abrir DDL da Camada Gold
    try:
        ddl_gold = open('gold_ddl.sql').read()
    except FileNotFoundError:
        print("Erro: Arquivo 'gold_ddl.sql' não encontrado.")
        print("Por favor, crie o arquivo .sql no mesmo diretório.")
        sys.exit(1)

    with connect(connection_string) as conn:
        with conn.cursor() as cur:
            
            # 3.1. Executar DDL (Limpar e Recriar Schema Gold)
            print("Executando DDL da camada Gold (Limpando e recriando tabelas)...")
            cur.execute(ddl_gold)
            print("Schema 'gold' e tabelas recriados.")

            # 3.2. Carga DIM_HOST (Chave Natural)
            print("Iniciando carga: DIM_HOST")
            cols_host = ['host_id', 'host_name', 'host_identity_verified', 'calculated_host_listings_count']
            df_host = df[cols_host].drop_duplicates(subset=['host_id'])
            
            insert_query = sql.SQL("INSERT INTO gold.DIM_HOST (id_host, host_name, host_identity_verified, calculated_host_listings_count) VALUES (%s, %s, %s, %s) ON CONFLICT (id_host) DO NOTHING")
            data_tuples = [tuple(x) for x in df_host.to_numpy()]
            cur.executemany(insert_query, data_tuples)
            print(f"DIM_HOST carregada: {len(df_host)} registros únicos enviados.")

            # 3.3. Carga DIM_LOCALIZACAO (Chave Surrogada)
            print("Iniciando carga: DIM_LOCALIZACAO")
            cols_loc = ['lat', 'long', 'neighbourhood', 'neighbourhood_group']
            df_loc = df[cols_loc].drop_duplicates()
            
            insert_query = sql.SQL("INSERT INTO gold.DIM_LOCALIZACAO (lat, long, neighbourhood, neighbourhood_group) VALUES (%s, %s, %s, %s)")
            data_tuples = [tuple(x) for x in df_loc.to_numpy()]
            cur.executemany(insert_query, data_tuples)
            
            # Adicionar registro "Desconhecido" para FKs nulas
            cur.execute("INSERT INTO gold.DIM_LOCALIZACAO (lat, long, neighbourhood, neighbourhood_group) VALUES (NULL, NULL, NULL, NULL) RETURNING id_localizacao")
            unknown_loc_key = cur.fetchone()[0]
            print(f"DIM_LOCALIZACAO carregada: {len(df_loc)} registros únicos + 1 (Desconhecido).")

            # 3.4. Carga DIM_PROPRIEDADE (Chave Surrogada)
            print("Iniciando carga: DIM_PROPRIEDADE")
            cols_prop = ['name', 'room_type', 'minimum_nights', 'cancellation_policy', 'instant_bookable', 'construction_year', 'has_house_rules']
            df_prop = df[cols_prop].drop_duplicates()
            
            insert_query = sql.SQL("INSERT INTO gold.DIM_PROPRIEDADE (name, room_type, minimum_nights, cancellation_policy, instant_bookable, construction_year, has_house_rules) VALUES (%s, %s, %s, %s, %s, %s, %s)")
            data_tuples = [tuple(x) for x in df_prop.to_numpy()]
            cur.executemany(insert_query, data_tuples)
            
            # Adicionar registro "Desconhecido" para FKs nulas
            cur.execute("INSERT INTO gold.DIM_PROPRIEDADE (name, room_type, minimum_nights, cancellation_policy, instant_bookable, construction_year, has_house_rules) VALUES (NULL, NULL, NULL, NULL, NULL, NULL, NULL) RETURNING id_propriedade")
            unknown_prop_key = cur.fetchone()[0]
            print(f"DIM_PROPRIEDADE carregada: {len(df_prop)} registros únicos + 1 (Desconhecido).")

            # 3.5. Carga DIM_ULTIMA_AVALIACAO (Chave Surrogada)
            print("Iniciando carga: DIM_ULTIMA_AVALIACAO")
            # Converte a coluna no DF principal ANTES de qualquer coisa
            df['last_review'] = pd.to_datetime(df['last_review']) 
            
            df_aval = df[['last_review']].drop_duplicates().dropna() # dropna() remove datas nulas

            # Derivar atributos de tempo
            df_aval['ano'] = df_aval['last_review'].dt.year.astype('Int64')
            df_aval['mes'] = df_aval['last_review'].dt.month.astype('Int64')
            df_aval['trimestre'] = df_aval['last_review'].dt.quarter.astype('Int64')

            insert_query = sql.SQL("INSERT INTO gold.DIM_ULTIMA_AVALIACAO (last_review, ano, mes, trimestre) VALUES (%s, %s, %s, %s)")
            data_tuples = [tuple(x) for x in df_aval.to_numpy()]
            cur.executemany(insert_query, data_tuples)

            # Adicionar registro "Desconhecido" para FKs nulas (NaN/NaT)
            cur.execute("INSERT INTO gold.DIM_ULTIMA_AVALIACAO (last_review, ano, mes, trimestre) VALUES (NULL, NULL, NULL, NULL) RETURNING id_avaliacao")
            unknown_aval_key = cur.fetchone()[0]
            print(f"DIM_ULTIMA_AVALIACAO carregada: {len(df_aval)} registros + 1 (Desconhecido).")

            # --- 4. MAPEAMENTO DE CHAVES (MERGE) ---
            
            print("\nIniciando mapeamento de Chaves Surrogadas...")
            # Ler dimensões de volta com as chaves geradas (BIGSERIAL)
            df_loc_com_chaves = pd.read_sql("SELECT * FROM gold.DIM_LOCALIZACAO", conn)
            df_prop_com_chaves = pd.read_sql("SELECT * FROM gold.DIM_PROPRIEDADE", conn)
            df_aval_com_chaves = pd.read_sql("SELECT * FROM gold.DIM_ULTIMA_AVALIACAO", conn)

            # --- CORREÇÃO APLICADA AQUI ---
            # Garante que a coluna lida do DB (que pode ser 'object' por causa do NULL)
            # tenha o mesmo tipo do DF principal ('datetime64[ns]')
            df_aval_com_chaves['last_review'] = pd.to_datetime(df_aval_com_chaves['last_review'])
            # --- FIM DA CORREÇÃO ---

            # Merge para mapear chaves surrogadas no DF principal
            df = pd.merge(df, df_loc_com_chaves.drop_duplicates(subset=cols_loc), on=cols_loc, how='left')
            df = pd.merge(df, df_prop_com_chaves.drop_duplicates(subset=cols_prop), on=cols_prop, how='left')
            df = pd.merge(df, df_aval_com_chaves.drop_duplicates(subset=['last_review']), on='last_review', how='left')
            
            # Preencher FKs nulas com a chave "Desconhecido" para garantir NOT NULL
            df['id_localizacao'] = df['id_localizacao'].fillna(unknown_loc_key).astype(int)
            df['id_propriedade'] = df['id_propriedade'].fillna(unknown_prop_key).astype(int)
            df['id_avaliacao'] = df['id_avaliacao'].fillna(unknown_aval_key).astype(int)
            print("Mapeamento de chaves concluído.")
            
            # --- 5. LOAD FATO_ANUNCIO ---
            
            print("Iniciando carga: FATO_ANUNCIO")

            # Renomear colunas para corresponder às FKs
            df = df.rename(columns={
                'host_id': 'FK_DIM_HOST_id_host',
                'id_localizacao': 'FK_DIM_LOCALIZACAO_id_localizacao',
                'id_propriedade': 'FK_DIM_PROPRIEDADE_id_propriedade',
                'id_avaliacao': 'FK_DIM_ULTIMA_AVALIACAO_id_avaliacao'
            })

            cols_fato = [
                'availability_365', 'price', 'service_fee', 'number_of_reviews', 
                'reviews_per_month', 'review_rate_number', 
                'FK_DIM_HOST_id_host', 'FK_DIM_LOCALIZACAO_id_localizacao', 
                'FK_DIM_ULTIMA_AVALIACAO_id_avaliacao', 'FK_DIM_PROPRIEDADE_id_propriedade'
            ]
            
            df_fato = df[cols_fato]
            
            insert_query = sql.SQL("""
                INSERT INTO gold.FATO_ANUNCIO (
                    availability_365, price, service_fee, number_of_reviews, 
                    reviews_per_month, review_rate_number, 
                    FK_DIM_HOST_id_host, FK_DIM_LOCALIZACAO_id_localizacao, 
                    FK_DIM_ULTIMA_AVALIACAO_id_avaliacao, FK_DIM_PROPRIEDADE_id_propriedade
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """)
            
            data_tuples = [tuple(x) for x in df_fato.to_numpy()]
            cur.executemany(insert_query, data_tuples)
            print(f"FATO_ANUNCIO carregada: {len(df_fato)} registros.")

            # Commit final
            conn.commit()
            print("\n--- Processo ETL para Camada Gold CONCLUÍDO com sucesso! ---")

except psycopg.Error as e:
    print(f"\n--- Ocorreu um erro de psycopg no LOAD ---")
    print(f"Erro: {e}")
    sys.exit(1)
except Exception as e:
    print(f"\n--- Ocorreu um erro inesperado no LOAD ---")
    print(f"Erro: {e}")
    sys.exit(1)

--- Iniciando processo de Extract (Camada Silver) ---
Conexão estabelecida.
Dados carregados da Silver. Total de 99449 linhas.

--- Iniciando processo de Transform & Load (Camada Gold) ---
Executando DDL da camada Gold (Limpando e recriando tabelas)...
Schema 'gold' e tabelas recriados.
Iniciando carga: DIM_HOST
DIM_HOST carregada: 99448 registros únicos enviados.
Iniciando carga: DIM_LOCALIZACAO
DIM_LOCALIZACAO carregada: 65426 registros únicos + 1 (Desconhecido).
Iniciando carga: DIM_PROPRIEDADE
DIM_PROPRIEDADE carregada: 93568 registros únicos + 1 (Desconhecido).
Iniciando carga: DIM_ULTIMA_AVALIACAO
DIM_ULTIMA_AVALIACAO carregada: 2436 registros + 1 (Desconhecido).

Iniciando mapeamento de Chaves Surrogadas...
Mapeamento de chaves concluído.
Iniciando carga: FATO_ANUNCIO
FATO_ANUNCIO carregada: 99449 registros.

--- Processo ETL para Camada Gold CONCLUÍDO com sucesso! ---
